In [ ]:
'''
This part of the code loads the key libraries.
If there is an error here, it is most likely a library needs to be installed.
'''

import io
import os, re, urllib3, requests, datetime
import pandas as pd
import numpy as np
from datetime import date
from bs4 import BeautifulSoup
import camelot
from dateutil.parser import parse
import PyPDF2
import warnings
#warnings.simplefilter("ignore", category=pd.errors.SettingWithCopyWarning)

'''
Defines folders.
The variable "d" captures the current folder where all programs are saved.
/source/ is where raw data is stored
/data/ is where the clean dataset is stored
folder_spec saves downloaded files (pdfs)
'''
d = os.getcwd()+"/"
PATH_RAW = f"{d}/../source"
PATH_FORMAT = f"{d}/../data"
folder_spec = PATH_RAW

'''
When connecting to a website, it is a good idea to mask the connection.
hdr masks the connection. It informs the website that you are connecting with a Browser. This reduces errors.
'''
hdr = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537'}


'''
The extract_page_text function opens a pdf an reads all the text. It is used to search specific pages
'''
def extract_page_text(pdf_path):
    with open(pdf_path, 'rb') as pdf_file:
        reader = PyPDF2.PdfReader(pdf_file)
        page_texts = [page.extract_text() for page in reader.pages]
    return page_texts

'''
Connecting to USA FRED requires an API Key. This key allows access to automatically download series.
Request FRED API key here: https://fred.stlouisfed.org/docs/api/api_key.html
'''
api_key = "6082258bcc5fa73a032d9d60c890f744"

# www.bahamas.gov.bs

- Website is undergoing changes https://www.bahamas.gov.bs/investment-statistics

## GDP

In [ ]:
# Define the website to be extracted.

u = "https://www.bahamas.gov.bs"
url = f"{u}/wps/portal/public/key%20statistics/economics%20statistics/national%20accounts/!ut/p/b1/vZfbkps4EIafJQ_gWELi4EvOxpgzwoYbypgx5mQYwAb76ZdJzUWySWa2ahOjK6r-rq_0q7vVWkbL_TK6HG55dhjy5nKo3v4jJnYE4HjbFeI5ScaAZzDPCrSLVMzMgnAWgN98PPgWj4Bq8DzmDJUGDNCIveVtdk1ZOrXcLfdByExSY4yS7MbnkEzeq-CMQ2FnOtGL0vbNDOfJ635x8gDVXgZkV_UlegCJO4a2nZKa3otrX3vIQ1jnFqM6E4eGi-gNx9dMtzCjaLR_6Oyyu-x9sI8zFgX8gvZc7JXlWKJhL-VCnGvXnRANCdhZG30CVjO8-KxPUaMD9AxWJKIjlSI9-3oVE5y2Ui9QhjVcLqzDtU0irt99-GCjH_kA6U_iMfh_8bPgg3MECnyP_0Dwn_Lgt_HUZ_G7ZfThFt8c-Cb4KNU-S7boYxc-FcCluW7ql2U4y9jvQH4AgAYB73rQAZwJl_5yD3DsFfdWe5QPtwCTZZQc6Mv8ASWCDElyTVJSZqqYXmEhKEPd3BrAZ52xf7iWmSZ2GrhE4CVFl1nl30CL8lczUISMTmjgAfy3gSptcbOVrG_zNAVUCz4biJ5qqWpb4K_v8Iekwc6fP8PNMsqT-ut4rL-Cr9yKgTTL4RXHAIw5dhkUIbOSem2UNfPUkkF9KV4eAcnHTDhnugKd-oxLcmPIukTsqnQr5liUjzjKZpr9cB-Iz_b1Rb35V9goGyRQsOHEcHyVRlm90-LOcwTCldSpCdos2jXqnokVemER0U_X5WsqpAQRnSDlmiTJ8WbUXZs0UVtoYswiDWorl9wbFJZZn6SKnWBKN8oSOedjxdBo6kZpSDOpT1VmofGYvfNfvrwb_LviDX51orrhE8rwQwp2BBjg4ZmkssxBhaYUWZBQD0PaWIa_0Y2HejfM9bvBwrm80e5nQPhsIHouUP1l3_mrQObZQPrJQBU8G0g9G_jny-KHRgdpTGPAMADMLY8C1FujA5PUG5kk725lHxTZIr2i_ZU35UEZBUXLBBVvErjqDtmasb0TuomL3XEsctPpNIsPQ0wnlzO8jKJ1AZnShWPcdEFBB_wG7DVPr-2d7HCjWK-rOC6mDKqUUklTQN29eebMT9dzlFeLDKhF7i9eVjQwo67Rk30V3cLWvh3IxT8MXZ0MQbOp4Jky-2vkv8Qs6_THRWKknxjK_VwU843gkxJ6ZQ8ggKYBCs_0Fc0YPGhIh_t8c2wN30EGgFb_EDbD4L8bKsrbXECfAdknAy38bCB8NvDZluI_b-kPVYg5jCkOIrCiWJbmuG_jBvs2bkhyEL-NG0w_9ex64kNZudaDrm007K3iPdoJwVZwLL9gmFuX4fCqIWNxuAPQX-_Dltl2h12zElrpdX7QOR3ybH97r1rkVWedndbSKdyR0SuPh3W7Oh68BdqIJ3LFd6VhHFMiVCAEcbSbqhXI7-nVTk9JceMu4uQq0r30It47z2_f1qum8ymWldUpsLdNTrelHtI8mx6VhBzNEEeMO9GmzWOZrhuJjpxtTd19A3LztOJ7NBrncaStyU2fvy3jyi5nnr5bdPfTun0vMP4Bg8DhPA!!/dl4/d5/L2dBISEvZ0FBIS9nQSEh/"
print(url)
r = requests.get(url=url, verify=None).content

# BeautifulSoup reads and processes the webiste portal
soup = BeautifulSoup(r)

# Filter all hyperlinks (<a>) in the webiste and extract those that are pdfs with the title "quarterly"
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links = [ link for link in links if ".pdf" in link and "quarterly" in link.lower() and "press" not in link.lower() ]

# The latest quarterly report is the first one in hte list:
get = f"{u}/{links[0]}"
# Access the report
response = requests.get(get, stream=True)

'''
This chunk of code downloads the target pdf
It is saved as gdp_file.pdf in the folder specified with folder_spec
'''
file_name = f"{folder_spec}/gdp_file.pdf"
with open( file_name , "wb") as pdf_file:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            pdf_file.write(chunk)

'''
Now, read the tables inside the pdf.
the function extract_page_text browses teh pdf and extracts everything. We then look for the words that define the target table
'''
tables = camelot.read_pdf( file_name , pages="all")

def extract_page_text(pdf_path):
    with open(pdf_path, 'rb') as pdf_file:
        reader = PyPDF2.PdfReader(pdf_file)
        page_texts = [page.extract_text() for page in reader.pages]
    return page_texts

page_texts = extract_page_text(file_name)
table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "TABLE 2" in text]

'''
Iterate over all pages with the word TABLE 2 and extract them as tables inside a list "tables"
'''
if table_2_pages :
    tables = []
    for page in table_2_pages:
        page_tables = camelot.read_pdf(file_name, pages=str(page))
        tables.extend(page_tables)

# create an empty dataframe with the results
df0 = pd.DataFrame()

'''
This loops iterates on the tables and cleans them.
The process: 1) finds the headers in the table; 2) looks for rows of interest with the words ISIC and Industry; 3) reshapes/melt the data; 4) manage dates and numbers
'''
for x in range(0,len(tables)) :

    # Open the Table
    df = tables[x].df

    # Find the row with headers
    header_row_index = df.apply(lambda row: row.astype(str).str.lower().str.contains("industry classification").any(), axis=1).idxmax()
    df.columns = df.iloc[header_row_index]
    df = df[(header_row_index + 1):]
    df.columns = df.columns.str.lower()

    # rename columns and define the one of interest
    for col in df.columns :
        if "isic" in col :
            df.drop(col, axis=1, inplace=True)
        elif "industry" in col :
            df[col] = df[col].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.lower()
            df.rename(columns = {col : 'industry'}, inplace=True)

    # Reshape the dataframe
    df = pd.melt(df, id_vars="industry", var_name="date", value_name='gdp')
    # Transform into numeric and manage commas
    df.gdp = pd.to_numeric(df.gdp.str.replace(',',''), errors='coerce')
    # Drop missing values in gdp.
    df.dropna(subset='gdp', inplace=True)
    # Manage dates
    df.date = df.date.str.replace(' r','').str.replace(' p','').str.strip().str.lower().str.replace('qq','q').str.replace(r'(q\d) (\d+)', r'\2-\1', regex=True)
    df['date'] = pd.to_datetime(pd.PeriodIndex(df['date'], freq='Q').to_timestamp() + pd.tseries.offsets.QuarterEnd(0))
    df.set_index('date', inplace=True)
    df.loc[df['industry'].str.contains('gdp by'), 'industry'] = 'gdp'
    df = df[df.industry == 'gdp']
    df = df[['gdp']]

    # Merge into one dataframe
    df0 = pd.concat([df0, df], axis=0)

print(f"Last date: {df0.index.max()}")

df0.to_csv(f"{PATH_RAW}/gdp.csv")
os.remove(f"{file_name}")

## CPI

In [ ]:
# Define the page of interest and access
url = "https://www.bahamas.gov.bs/wps/portal/public/key%20statistics/economics%20statistics/consumer%20price%20index%20(cpi)/!ut/p/b1/vZbZjqM6EIafpR-AxuxwCYGwJOxLgBsEIRB2GggQnv5kjkY6OpqZzs0I15VLf-lz_bZlwyHsw2Ebz0UeT0XXxvWPeUhGJgdM-8xgLM0LOGBJnKU4wsJEnIQvsO8FJMOPas7LVna3PVsHpnm3-ItRxU-PtVSyTuiDWHpzi07JsR1p7NZWUXvy1ilJcghKD3KVUdde64pJAAVlHfhaVmabrcWLcZo17bRoyQ0T2pJ0D24GcQnkcIztsmPlKF6ZseZw5mywlCwJUaK8-WXDVCPQ-WJ40Ek-6gMhDzRvjFBUP4QqFtuOKb3HTIiON8huyI0XPpsoqJVuJ5P9-Hg1HbyaBn8YLPjOE3BEftZ_I3jj6Zt69P2ehP9KMCCqLIvTqkgAEsiucWYNSkJ1HPwUfNfi902i7wQIHLwE1B8FHoAd2Ad4ZJfPXt6qzSo3c1EBoqml-loCctKqyNXcWtcmTtPOlvzK6RqvaXYSAIeyZHXKjNSzXI7lp1VKoXdAZG8gti9QBPjeQHJvILEzUAR7A9G9gX__WihwWCTN53JtPsEnQuAEDkgSAJohUYDCnhI8Gb5TWIENMy7oJzRGJtVPhCUPp00Srze2xSpIO_MTeSu9YGsuCk7oV__JXe2q7geCzEzUsmRm4MXStn1NMVQuD6W0eKWTXM5mpOJwuloiDKVmKF0P0zkVe78GCAe0r8qJDDNnonxwCcJPB5Tjc-Icjh-wJnXN7Y1d9G-OvKWoYNLULTXA5q5OKXPCPe4f97hZq4YMi7h5XpLmmMfKvQsbaGj0ouvY5_Fr85fXa_MGSO0M1PG9gcjewL0txf--pf-7YziN4yiNYIBBKYqgadgrA2hNRnXhBSuyggebNkX38GYs6w5xH3K1XoRFETjCqlLZfSlIkS1ODr9ceTzFBP3Yer1nSNgaIJHHmltdnUtiDOM25dCKhkq2cfzOYQ7HzYuX04pnwfEkG_c4ywyGlsZp_GruveVO2usDZijp5ZEPvT0xquU_CUuKuieGmdBUBLPSXqZmXq3Rufd1erQkTFtfP8QvCVnjUZfscEwrAY_DfvTH8DgOYiN8Sd3TTg3yrvOSaHCGEAqjFVVKMjDMNcWut6T64U_fzKfTmbSETPs18Bj6L4KPfwAM6rDA/dl4/d5/L2dBISEvZ0FBIS9nQSEh/"
r = requests.get(url=url, verify=None).content

# Get the pdf links of interest
soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links = [ link for link in links if ".pdf" in link and "bnsi" in link.lower() ]

# Access the latest link and save the pdf
link = links[0]
link = f"https://www.bahamas.gov.bs{link}"
response = requests.get( link, verify=False, timeout=300, headers=hdr)
file_name = "cpi.pdf"
file_name = f"{folder_spec}/{file_name}"
with open( file_name , 'wb') as file: file.write(response.content)
print(f"Downloaded")

# Find pages that have the target word:
page_texts = extract_page_text(file_name)
table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "All Items Indices" in text]
table_2_pages = table_2_pages[0]

# Extract the table
page_tables = camelot.read_pdf(file_name, pages=str(table_2_pages), flavor='stream')
table = page_tables[0].df

# Use this to take the first n rows as headers.
table.columns = table.iloc[:5].apply(lambda col: ' '.join(col.astype(str)), axis=0)
# Clean-up variable names
table.columns = table.columns.astype(str).str.lower().str.strip()
for col in table.columns :
    if " cpi" in col : table.rename(columns = {col : 'cpi' }, inplace=True)

# Date processing
table['year'] = table['year'].astype('str').str.strip()
table['year'] = pd.to_datetime(table['year'] , format="%Y", errors='coerce')
table['year'] = table['year'].dt.year.astype('Int64')

# Convert months to numbers
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}
table['month'] = table['month'].map(month_map).astype('Int64')

# Only the first month has the year. This code drags-on the last observed year to the next months
table['tyear'] = table['year'].bfill()
table.loc[table['month'] == 1, 'year'] = table['tyear']
table['year'] = table['year'].ffill()
table['year'] = table['year'].bfill()

# Transform to numeric data
table['cpi'] = pd.to_numeric(table['cpi'].astype(str).str.strip(), errors='coerce')
table['cpi'] = table['cpi'].bfill()

# Drop tows with empty months
table = table.dropna(subset='month')

# Generate a date variable
table['date'] = pd.to_datetime(table[['year', 'month']].assign(day=1))
table = table[['date', 'cpi']]
table.set_index('date', inplace=True)

# Save
print(f"Last date: {table.index.max()}")
table.to_csv(f"{PATH_RAW}/cpi.csv")
# Delete the extra file
os.remove(f"{file_name}")

## FISCAL

## TRADE

In [ ]:
df = pd.read_csv(f"{PATH_RAW}/bahamas imports.csv", parse_dates=True, index_col='date')

# Open the website
url = "https://www.bahamas.gov.bs/wps/portal/public/key%20statistics/economics%20statistics/trade%20and%20industry/!ut/p/b1/vZbZrptIFEW_JR_guIrZj8wGzEwxvVhmMGYyNthg-Po40W0p6ST3ttSJq56Q9tHS2Wco1vE6XMfnw1gWh1vZnQ_N1--Y2tscsN3dBmcZQSQASxEszZEOLhPUUxA9BeA3hwXf4nEg6yxLMLpMAgooyNqxFr3FTA1bB-vQR-RO6PRC3AFNJlNRUjCCFiQAiGkKbh3LXWNj3IIz3cybBFJ5cPNK8YQDdDaraEPlu_LIOwowKx6ddILZSko_lP2sujV_QmcegnBcpW6W7-1VfpsepIby1iBTLedce2Qr9TjffbSXJRZPQKCb-e4e8D5tQrDignSrgPiibkzcmWmUo7jQen_JAN7bV2w8VgIRPnZ69-nNh3cSfc8HSH4QT4D_F_8UvFNHIMG3-HcE_6kPfhuPfRQfrON3U_zqwDfBe632UbPF77vwoQCujW3X5uvoKaO_A3k-AAoErONCGzAGXHvrEBB7t5ovylIvTgUepl4zYKjLBQoI1wXBMVCNGZlkuJWJQxFqxk4HHm1Pw-KYRpZYme8gjhUkTaSlfwNNzNs8gTykNEQCFxB_GyiTJvO0kvYslsSAbMJXA_GXWipbJvjrGf7QNIT952uoruMyaT9PafsZfGY2FCRphtgwFCAIhl77VURthEGZRMU4XtBNzqt88VE5Fdyp0CRotyeiRiOFtjVOb2qnodKqXvZx8aRZi7PgbBG2Z3n07rCTVJzDYMfw0XQVJlGeST5wbQ4xNXbs_EsRB50cUnuJXJmI97Jtfc24DOFIQ7h0T5IkHfW2vyRdfKkUfk_jClQ2Dpo7PKqLIckkKyEwTa9r3D6lDUXij34SblkhDJlMrRSWoGf206c3g383vP6vKqrpHsJ0L8Jgj4AOFtdAjWncZGgIsQkRtuiCauqequmLPOvG9s1g7lSPpPMREL4aiL8WKP9y7_xVIPVqIPlioAxeDcReDfzzY_HDooMkQRKAogB4rjwMYF8XHXgIg14IYjDWg18Vq-yOh3fWEG_SxElKwcmEmsBNfyi2lOUe8ZFfBelUlYbdKyYbRQSZnE_wPPHmGRRSH037rvcr0mdVECqu1lqBaDMT326b_b56FFDGpEZ4-Njs9uewPN5PcdmsCiBXpbfKNyQw4r7TkrCJx-hijQd09g63vk1ufqc28IQZwz328j1N20O6SvTsA0OZn4eCdqfh-Vh49UWHFcKNPkGGJwIjUw29kizoBZi3sydX7DADdQ_j5v3zcizjNRo-AtIvBprEq4Hw1cBXW0r8eUt_mEKCIQiMgTjYYDRNMswvfjcu5xjbtuyk7FxS7Kq2mVFo7fNzY0noWgv1jCdDRUwMV0YM9C-XOGsOSA8DkDmUF5hX95rxeNuMQbgLwKHZuqH8sIR0SS9Fo9BoxCkS1DJQOe-sZyOPMk1eXRPNpor0rM6rQxwbuJUecZbI5pMhDVGOyn0YYaab1NYD3YNl2u5bPkyBINVbyzdKs2t0nCuzwWhiIh5GQT24-cQ9nEmdT4XQcHJ-fE7tncbITb2-tGjUnmdHOaLDGMfvLtn_dMfvBfoX_dCRfg!!/dl4/d5/L2dBISEvZ0FBIS9nQSEh/"
r = requests.get(url=url, verify=None).content
soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links = [ link for link in links if ".pdf" in link and "quarter" in link.lower() ]

'''
Find the year and quarter in the name of pdf files.
Extract year/quarter from the name of files.
filtered_links keep only the files that have a year/quarter above the last recorded data point in "df"
'''

def extract_year(url):
    match = re.search(r'(20\d{2})', url)
    if match:
        return int(match.group(1))
    return None
links = [link for link in links if extract_year(link)>= df.index.max().year]


def extract_quarter_year(url):
    match = re.search(r'(\d{1,2})(st|nd|rd|th)\+quarter\+(\d{4})', url.lower())
    if match:
        quarter = match.group(1)
        year = match.group(3)
        return quarter, year
    return None, None

links = [ link for link in links if "Press" not in link]
filtered_links = []
for link in links :
    ext = extract_quarter_year(link.split("/")[-1])
    thedate = f"{ext[1]}-Q{ext[0]}"
    thedate = pd.Period(thedate, freq='Q').to_timestamp() + pd.tseries.offsets.QuarterEnd(startingMonth=0)
    if thedate > df.index.max() : filtered_links.append(link)

In [ ]:
'''
# OMIT. USE ONLY IF HISTORIC IS REQUIRED.
pages = [ link.get("href") for link in soup('a') if link.get("href") is not None and link.get('id') is not None and 'Page_' in link.get('id') ]
for page in pages :
    u = f"https://www.bahamas.gov.bs/{page}"
    r = requests.get(url=u , verify=False).content
    
    soup = BeautifulSoup(r)
    morelinks = [ l.get("href") for l in soup('a') if l.get("href") is not None ]
    morelinks = [ l for l in morelinks if ".pdf" in l and "quarter" in l.lower() ]
    for l in morelinks : links.append(l)
'''

# Iterate on all filtered_links
if len(filtered_links)>0:
    df0 = pd.DataFrame()
    
    for get in links :   
        # Open files and save them in PATH_RAW
        u = f"https://www.bahamas.gov.bs/{get}"
        response = requests.get(u, verify = None)
        if response.status_code == 200:
            file_name = u.split("/")[-1].replace('?MOD=AJPERES','')
            file_name = f"{PATH_RAW}/{file_name}"
            with open(file_name, 'wb') as file:
                file.write(response.content)
            print(file_name)
        else :
            print(f"skipped: {u}")
            continue

        # Extract the Table from page 10.
        dfi = tabula.read_pdf(file_name, pages=10, pandas_options={'header': None})[0]

        # Column names are in the column that contains the text QTR
        l = dfi.loc[dfi[dfi.columns[-3]].str.contains('QTR')==True].index[0]
        col = (dfi.iloc[l] + dfi.iloc[l+1])
        col[1] = 'variable'

        # Minor cleaning in variable names
        dfi.columns = col
        dfi.columns = dfi.columns.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.lower()
        dfi.columns = dfi.columns.str.replace('qtr','').str.replace('st','-').str.replace('nd','-').str.replace('rd','-').str.replace('th','-')
        dfi.columns = dfi.columns.str.replace(' ','')
        dfi.variable = dfi.variable.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.lower().str.strip()
        dfi['variable'] = dfi['variable'].str.replace(r'(?<=^\w) (?=\w)', '', regex=True)
        for i in range(1, len(dfi)):
            if dfi.loc[i, "variable"] == "materials":
                # Merge the current materials row with the one above
                dfi.loc[i, "variable"] = f"{dfi.loc[i - 1, 'variable']} {dfi.loc[i, 'variable']}"
        # Drop Missing values
        dfi.dropna(subset='variable', axis = 0, inplace=True)
        dfi = dfi.loc[:, dfi.columns.notna()]

        # Reshape
        dfi = pd.melt(dfi, id_vars="variable", var_name="date", value_name='imports')
        dfi.dropna(subset='imports', inplace=True)
        
        # Find the quarter in the date.
        dfi.date = dfi.date.str.strip().str.replace(' ', '').str.replace(r'(\d)-(\d+)', r'\2-Q\1', regex=True)
        dfi.date = pd.to_datetime(pd.PeriodIndex(dfi.date, freq='Q').to_timestamp() + pd.tseries.offsets.QuarterEnd(startingMonth=0))
        dfi.set_index('date', inplace=True)

        # Transform into numeric variables
        dfi.imports = dfi.imports.str.replace(',','')
        dfi.variable = dfi.variable.str.replace('miscellneous','miscellaneous')
        dfi.imports = pd.to_numeric(dfi.imports, errors = 'coerce')
        dfi.dropna(subset='variable', axis = 0, inplace=True)
        dfi.dropna(subset='imports', axis = 0, inplace=True)

        # Reshape: each column is a variable
        dfi = dfi.pivot(columns='variable', values='imports')
        dfi = dfi.rename(columns=lambda x: 'imports ' + str(x))
        dfi.sort_index(inplace=True)

        # Merge into the df0 dataframe
        df0 = pd.concat([df0, dfi], axis=0)
        df0.sort_index(inplace=True)
        # Delete the pdf file
        os.remove(file_name)
    
    df = pd.concat([df,df0], axis=0).sort_index()
    df = df[~df.index.duplicated(keep='first')]
    print(f"Updated: {df.index.max()}")
    df.to_csv(f'{PATH_RAW}/bahamas imports.csv')
else :
    print("Nothing to update")

In [ ]:
# Open the website
url = "https://www.bahamas.gov.bs/wps/portal/public/The%20National%20Budget/Budget%20Performance%20Reports/!ut/p/b1/vZLZcptAEEW_RR8gMzPsj4hNLAOIHV4otIAWI8AgkPj6SEkqiyux8xB7-mmq7p3TfXuIlIiJ9JwPhzLvD_U5f37cUyYjVRuvRIrDHEPxQIO-pUmyQ6o8uguSuwD85Qjgmx-oWBDufpW2OaAFrO8INAKqCIiIiPWENKVaGxXZzZq0u1Q81YrrvSxTownckuwkhSdlw-oNNxj4udFO-Iy583geiryZKmswrHE4es4qE_QiN0vcOsuz3vgRxNT-lKF8R140Wo_0_Y4r9CWPxluG0ux06FXVlqvhtL2O0lHuleVxXu0iYbHfkHMrApi9SXF-IlUY3BLoL8RaFY1QSfVSlg3VhuTKKNqq4Bc7QRC6bvY9izeG_acsf_gBA9HDby5CnYfAQ-_5IyL9Pe4_vPBV8Na63luYtayrHZHcZewvMlOg773YtEkaCkQBTfhEDKjMO94abTpN7nHyRkvmED7VNwig7vmma_kKtrauhV9c7UHHkgA8udE9QE1WLznb0A0WgkDPqTZ8DbSR_xhehIwR0MCjwEcDXwVhwM8Gkp8aqSqBDwe--jQO-u9AnUgP6-pp3FRP4IlmAUVxiKF5xFIcTxHhMWFYsdNXkpYWTdCrfpSQiusv1Lm-ij23ryc4rpur2qJsCMKN0swz5xhfE6pVwqOHy9pkhz2T3sappmrzMg2lbRi-K0je8KKNzWq5vhnhdJuPEF0XStOnbdE0I3_CjR2f1dzk3X29rBh6e8nSoQkPU9SKKezxS0c-90W679IQSF11fWHixFtddY4sOJ-Zx2196Rt7Iu3KW-9y1LUePm_3YKehjsUOtemc6VDa4bBgPZIuSh3C5S4qZzOiqYLBMBl3OULuZ42zL-AiLcU!/dl4/d5/L2dBISEvZ0FBIS9nQSEh/"
r = requests.get(url=url, verify=None).content
soup = BeautifulSoup(r)

# The data is available as a xlsx in the site
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links = [ link for link in links if ".xlsx" in link ]
link = f"https://www.bahamas.gov.bs/{links[0]}"
r = requests.get(url=link, verify=None).content
# Read the excel.
df = pd.read_excel(r)

# Clean the columns
df.columns = df.columns.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.lower()
for col in df.columns :
    if "unnamed: 0" in col : df.rename(columns = {col : 'variable'}, inplace=True)
df.variable = df.variable.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.lower().str.strip()
df = df.dropna(subset='variable')

# This part of the code select variables of interest.
df = df[
    df['variable'].str.startswith("tax revenue") |
    df['variable'].str.startswith("a.") |
    df['variable'].str.startswith("b.") |
    df['variable'].str.startswith("c.") |
    df['variable'].str.startswith("d.") |
    df['variable'].str.startswith("e.") |
    df['variable'].str.startswith("f.") |
    df['variable'].str.startswith("g.") |
    df['variable'].str.startswith("h.") |
    df['variable'].str.startswith("j.") |
    df['variable'].str.startswith("i. capital") |
    df['variable'].str.startswith("i. miscel") |
    (df['variable'] == 'i.  general taxes on goods & services') |
    (df['variable'] == 'value added tax') |
    (df['variable'] == 'ii.  taxes on specific services') |
    (df['variable'] == 'iii. taxes on use/permission to use') |
    (df['variable'] == 'total revenue') |
    (df['variable'] == 'total expenditure')|
    (df['variable'] == 'surplus/deficit')
]
# Add fiscal to all variable names
df['variable'] = 'fiscal ' + df.variable
# Reshape
df = pd.melt(df, id_vars="variable", var_name="date", value_name='value')
df = df.dropna()
df.date = pd.to_datetime(df.date, infer_datetime_format=True, errors='coerce')
df.set_index('date', inplace=True)
df.loc[df['value']==0, 'value'] = np.nan
df = df.dropna()

df = df.pivot(columns='variable', values='value')

print(f"Updated: {df.index.max()}")
df.to_csv(f"{PATH_RAW}/bahamas mof.csv")

# tourismtoday

In [96]:
try :
    # Read the page and find all links to pdfs
    url = "https://www.tourismtoday.com/statistics/foreign-arrivals-air-sea-data"
    r = requests.get(url=url, verify=None, headers=hdr).content

except :
    from urllib3.util.retry import Retry
    from requests.adapters import HTTPAdapter
    from urllib.parse import urljoin
    hdr = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
    }

    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=[403, 429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    session.mount("https://", HTTPAdapter(max_retries=retries))

    resp = session.get(url, headers=hdr, timeout=30)   # <-- timeout matters
    print(resp.raise_for_status())
    r = resp.text


soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links = [ link for link in links if ".pdf" in link and "arrivals" in link.lower() and "air" in link.lower() and "sea" in link.lower() and "thru" not in link.lower() and "1998" not in link ]

In [94]:
'''
Updating the data requires only new data.
Start by opening the data that we have stored before under df_base
then, find the pdfs that only have years not recorded in df_base
Extract also the current year (because it updates every month)
'''
try :
    df_base = pd.read_csv(f"{PATH_RAW}/bahamas_arrivals.csv", parse_dates=True, index_col='date')
except :
    df_base = pd.DataFrame()
# get only the years not recorded in df_base
if df_base.index.max().month < 12 : last_year = df_base.index.year.max() - 1
else : last_year = df_base.index.year.max()

links = [ link for link in links if parse(re.sub(r'%\w{2}', '', link.split('/')[-1].replace(".pdf","").replace("%20",'_').replace("_0","").replace("_"," ")), fuzzy=True).year > last_year ]

print(f"To update: {len(links)}")

To update: 1


In [ ]:
# links is a list with all links with date greater than the last date extracted in the data.
# we now extract all pdfs corresponding to each link
for link in links :
    # urls may change.
    if "http" in link : u = link
    elif "www" in link : u = f"https://{link}"
    else : u = f"https://www.tourismtoday.com/{link}"

    # access the url
    response = requests.get(u, verify=False, timeout=300, headers=hdr)

    # Download the pdf
    file_name = u.split("/")[-1]
    file_name = f"{folder_spec}/{file_name}"
    with open( file_name , 'wb') as file: file.write(response.content)

    # Find the "summary report" page
    page_texts = extract_page_text(file_name)
    table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "Summary Report" in text]
    # Extract the year of the pdf. However, urls have special characters that must be removed first.
    year = parse(re.sub(r'%\w{2}', '', u.split('/')[-1].replace(".pdf","").replace("%20",'_').replace("_0","").replace("_"," ")), fuzzy=True).year
    # Extract all tables and add them to the list tables.
    tables = []
    for page in table_2_pages:
        page_tables = camelot.read_pdf(file_name, pages=str(page), flavor='stream')
        tables.extend(page_tables)

    # This is where the data will be saved.
    df0 = pd.DataFrame()
    # Iterate finding all tables
    for Table in tables :
        df = Table.df

        # columns in all pdfs come without a header. This function gives a name to variables.
        df.rename(columns = { 0 : 'island group', 1 : 'main island', 2 : 'island area',
                              3 : 'arrivals air', 4 : 'arrivals sea', 5 : 'arrivals cruise', 6 : 'arrivals total' }, inplace=True)
        
        header_row_index = df.apply(lambda row: row.astype(str).str.lower().str.contains("total").any(), axis=1).idxmax()
        #df.columns = df.iloc[header_row_index]
        df = df[(header_row_index + 1):]
        df.columns = df.columns.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.lower()

        # main island and island have missing values. Fill them up with the latest observation. We basically "drag" the last observation
        df['main island'] = df['main island'].replace('', np.nan).ffill()
        df['island area'] = df['island area'].replace('', np.nan)
        # drop missing values remaining:
        df = df.dropna(subset='island area')
        
        # Transform data into numeric
        for col in [ col for col in df.columns if "arrivals" in col ] :
            df[col] = pd.to_numeric(df[col].astype(str).str.replace("," , "") , errors='coerce')
        df = df.dropna(subset='arrivals total')

        # manage date using the year from the pdf title.
        df['date'] = pd.to_datetime(f"{year}-{tables.index( Table ) + 1 }-01", format="%Y-%m-%d")
        df.set_index('date', inplace=True)
        df = df[['main island', 'island area', 'arrivals air', 'arrivals sea', 'arrivals cruise', 'arrivals total']]
        df0 = pd.concat([df0, df], axis=0)

    # After extracting all tables, data is combined and prepared.
    # Pivot prep:
    df0 = df0[~df0['island area'].str.contains(r'\d', regex=True)]
    df0['variable'] = df0['main island'].str.cat(df0['island area'], sep=' ')
    df0 = df0[[ col for col in df0.columns if "arrivals" in col or "variable" in col ]]
    df0 = df0.reset_index().drop_duplicates(subset=['date', 'variable']).set_index('date')

    # Pivot creates a table with a variable per column
    df0 = df0.pivot(columns='variable').sort_index()
    df0.columns = [' '.join(col).strip() for col in df0.columns]

    df_base = pd.concat([df_base[df_base.index.year != df0.index.year.max()], df0] , axis=0)
    os.remove(file_name)

# Compute the totals.
df_base['totalarrivals'] = df_base.filter(like="arrivals total").sum(axis=1)
df_base['arrivals_airAllAll'] = df_base.filter(like="arrivals air").sum(axis=1)
df_base['arrivals_cruiseAllAll'] = df_base.filter(like="arrivals cruise").sum(axis=1)
df_base['arrivals_seaAllAll'] = df_base.filter(like="arrivals sea").sum(axis=1)
df_base.sort_index().to_csv(f"{PATH_RAW}/bahamas_arrivals.csv")

print(f"Updated: {df_base.index.max()}")

c:\Users\guerr\anaconda3\envs\est\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.tourismtoday.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Updated: 2025-10-01 00:00:00


C:\Users\guerr\AppData\Local\Temp\ipykernel_16376\4290788889.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['main island'] = df['main island'].replace('', np.nan).ffill()
C:\Users\guerr\AppData\Local\Temp\ipykernel_16376\4290788889.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['island area'] = df['island area'].replace('', np.nan)
C:\Users\guerr\AppData\Local\Temp\ipykernel_16376\4290788889.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.


# Central Bank

## CBOB/QSD

In [ ]:
# dataframe 'df' will store all the data.
df = pd.DataFrame()

# Open the website
url = "https://www.centralbankbahamas.com/publications/qsd?page=1"
r = requests.get(url=url, verify=None).content
soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links = [link for link in links if "pdf" in link]
#links = [link for link in links if "quarterly-statistical" in link]
links = list(set(links))

# Extract the date from the file name.
from datetime import datetime
def extract_date(url):
    match = re.search(r'/(\d{4}-\d{2}-\d{2})-', url)
    if match:
        return datetime.strptime(match.group(1), "%Y-%m-%d")
    return None

links = [(extract_date(url), url) for url in links]
link = max((d for d in links if d[0] is not None), key=lambda x: x[0])[1]

# Download the pdf
print(f"Latest: {link}")
response = requests.get( link, verify=False, timeout=300, headers=hdr)
file_name = link.split("/")[-1]
file_name = f"{folder_spec}/{file_name}"
with open( file_name , 'wb') as file: file.write(response.content)

print(f"Downloaded")
# page_text includes all the text in the pdf.
page_texts = extract_page_text(file_name)

In [ ]:
'''
The next series of code will find all tables.
The code extracts the page in which the table is.
Then read the page as a table, clean, and store in 'df'.
'''
# Table 2.30
table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "Table 2.30 " in text]
table_2_pages = table_2_pages[0]
page_tables = camelot.read_pdf(file_name, pages=str(table_2_pages), flavor='stream')
table = page_tables[0].df
table.columns = table.iloc[:4].apply(lambda col: ' '.join(col.astype(str)), axis=0)

table = table[[ col for col in table.columns if ( "arrears" in col.lower() and "over" in col.lower() and "30" in col.lower() ) or "period" in col.lower() ]]
table.columns = ['date' if 'period' in col.lower() else col for col in table.columns]
table['year'] = table['date'].astype(str).str.extract(r'(\d{4})')[0]
table['year'] = table['year'].ffill()
table.columns = table.columns.str.replace('[-\s]', '', regex=True).str.lower()
table = table.loc[:, ~table.columns.duplicated()]
table.columns = ['arrears30' if 'arrears' in col.lower() else col for col in table.columns]

table['date'] = table['date'].str.replace(".","")
table['date'] = pd.to_datetime(table['date'], format="%b", errors='coerce')
table = table.dropna(subset='date')
table['month'] = table['date'].dt.strftime('%m')
table['date'] = pd.to_datetime(table['year'].astype(str) + '-' + table['month'], format='%Y-%m')
table = table[['date', 'arrears30']].set_index('date')
df = pd.concat([ df, table ], axis=1)
print(f"Updated Arrears: {table.index.max()}")

# Table 2.28
table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "Table 2.28 " in text]
table_2_pages = table_2_pages[0]
page_tables = camelot.read_pdf(file_name, pages=str(table_2_pages), flavor='stream')
table = page_tables[0].df
table.columns = table.iloc[:2].apply(lambda col: ' '.join(col.astype(str)), axis=0)
table.rename(columns = { ' ' : 'variable' }, inplace=True)
table = table[table['variable'].str.lower().str.contains("net income ")]

table = pd.melt(table, id_vars="variable", var_name="date", value_name='banking_profit')
table['year'] = table['date'].str.extract(r'(\d{4}) Q').ffill()
table['quarter'] = table['date'].str.extract(r'(Qtr.\s?[I|II|III|IV]+)')
quarter_map = {'Qtr. I': 'Q1', 'Qtr. II': 'Q2', 'Qtr. III': 'Q3', 'Qtr. IV': 'Q4'}
table['quarter'] = table['quarter'].replace(quarter_map)
table = table.dropna( subset='quarter')

table.loc[table['quarter'] == 'Q1', 'year'] = np.nan
table['year'] = table['year'].bfill()
quarter_to_month = {'Q1': 3, 'Q2': 6, 'Q3': 9, 'Q4': 12}
table['month'] = table['quarter'].map(quarter_to_month)

table['date'] = pd.to_datetime(table.assign(day=1)[['year', 'month', 'day']], errors='coerce')
table = table.set_index('date')[['banking_profit']]
df = pd.concat([ df, table ], axis=1)
print(f"Updated Banking Profit: {table.index.max()}")

# Table 3.6
table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "Table 3.6 " in text]
table_2_pages = table_2_pages[0]
page_tables = camelot.read_pdf(file_name, pages=str(table_2_pages), flavor='stream')
table = page_tables[0].df
table.columns = table.iloc[:4].apply(lambda col: ' '.join(col.astype(str)), axis=0)
table.columns = table.columns.astype(str).str.lower().str.strip()
for col in table.columns :
    if "index" in col : table.rename(columns = {col : 'bisx_index' }, inplace=True)
    elif "listed" in col : table.rename(columns = {col : 'bisx_no' }, inplace=True)
    elif "volume" in col : table.rename(columns = {col : 'bisx_volume' }, inplace=True)
    elif "value" in col : table.rename(columns = {col : 'bisx_value' }, inplace=True)
    elif "period" in col : table.rename(columns = {col : 'date' }, inplace=True)
table['year'] = table['date'].str.extract(r'(\d{4})').ffill()

table['date'] = table['date'].str.replace(".","")
table['month'] = pd.to_datetime(table['date'], format="%b", errors='coerce')
table = table.dropna( subset='month')

table['month'] = table['month'].dt.strftime('%m')
table['date'] = pd.to_datetime(table['year'].astype(str) + '-' + table['month'], format='%Y-%m')
table.set_index("date", inplace=True)
table = table[ [ col for col in table.columns if "bisx" in col ]]

df = pd.concat([ df, table ], axis=1)
print(f"Updated BISX: {table.index.max()}")

# Table 2.5
table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "Table 2.5 " in text]
table_2_pages = table_2_pages[0]
page_tables = camelot.read_pdf(file_name, pages=str(table_2_pages), flavor='stream')
table = page_tables[0].df
table.columns = table.iloc[:6].apply(lambda col: ' '.join(col.astype(str)), axis=0)
table.columns = table.columns.astype(str).str.lower().str.strip()

for col in table.columns :
    if "to government" in col : table.rename(columns = {col : 'Gov Credit' }, inplace=True)
    elif "to private sector" in col : table.rename(columns = {col : 'Priv Credit' }, inplace=True)
    elif "rest of public sector" in col : table.rename(columns = {col : 'Rest Pub Sect Credit' }, inplace=True)
    elif "period" in col : table.rename(columns = {col : 'date' }, inplace=True)


table = table[['date', 'Gov Credit', 'Priv Credit', 'Rest Pub Sect Credit']]

table['year'] = table['date'].str.extract(r'(\d{4})').ffill()

table['date'] = table['date'].str.replace(".","")
table['month'] = pd.to_datetime(table['date'], format="%b", errors='coerce')
table = table.dropna( subset='month')

table['month'] = table['month'].dt.strftime('%m')
table['date'] = pd.to_datetime(table['year'].astype(str) + '-' + table['month'], format='%Y-%m')
table.set_index("date", inplace=True)

table = table[['Gov Credit', 'Priv Credit', 'Rest Pub Sect Credit']]

for col in table.columns :
    table[col] = pd.to_numeric(table[col].astype(str).str.strip().str.replace(",",""))

df = pd.concat([ df, table ], axis=1)
print(f"Updated Credit: {table.index.max()}")

# Table 2.7
table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "Table 2.7 " in text]
table_2_pages = table_2_pages[0]
page_tables = camelot.read_pdf(file_name, pages=str(table_2_pages), flavor='stream')
table = page_tables[0].df

table.columns = table.iloc[1:8].apply(lambda col: ' '.join(col.astype(str)), axis=0)
table.columns = table.columns.astype(str).str.lower().str.strip()

for col in table.columns :
    if "m1" in col : table.rename(columns = {col : 'm1' }, inplace=True)
    elif "m2" in col : table.rename(columns = {col : 'm2' }, inplace=True)
    elif "m3" in col : table.rename(columns = {col : 'm3' }, inplace=True)
    elif "period" in col : table.rename(columns = {col : 'date' }, inplace=True)

table['year'] = table['date'].str.extract(r'(\d{4})').ffill()

table['date'] = table['date'].str.replace(".","")
table['month'] = pd.to_datetime(table['date'], format="%b", errors='coerce')
table = table.dropna( subset='month')

table['month'] = table['month'].dt.strftime('%m')
table['date'] = pd.to_datetime(table['year'].astype(str) + '-' + table['month'], format='%Y-%m')
table.set_index("date", inplace=True)
table = table[['m1', 'm2', 'm3']]
df = pd.concat([ df, table ], axis=1)
print(f"Updated Monetary: {table.index.max()}")

# Table 1.1
table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "Table 1.1 " in text]
table_2_pages = table_2_pages[0]
page_tables = camelot.read_pdf(file_name, pages=str(table_2_pages), flavor='stream')
table = page_tables[0].df

table.columns = table.iloc[1:4].apply(lambda col: ' '.join(col.astype(str)), axis=0)
table.columns = table.columns.astype(str).str.lower().str.strip()

for col in table.columns :
    if "total external reserves" in col : table.rename(columns = {col : 'Reserves' }, inplace=True)
    elif "period" in col : table.rename(columns = {col : 'date' }, inplace=True)

table['year'] = table['date'].str.extract(r'(\d{4})').ffill()

table['date'] = table['date'].str.replace(".","")
table['month'] = pd.to_datetime(table['date'], format="%b", errors='coerce')
table = table.dropna( subset='month')

table['month'] = table['month'].dt.strftime('%m')
table['date'] = pd.to_datetime(table['year'].astype(str) + '-' + table['month'], format='%Y-%m')
table.set_index("date", inplace=True)

table = table[['Reserves']]

df = pd.concat([ df, table ], axis=1)

# Table 4.2
table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "Table 4.2 " in text]
table_2_pages = table_2_pages[0]
page_tables = camelot.read_pdf(file_name, pages=str(table_2_pages), flavor='stream')
table = page_tables[0].df

table.columns = table.iloc[1:5].apply(lambda col: ' '.join(col.astype(str)), axis=0)
table.columns = table.columns.astype(str).str.lower().str.strip()

for col in table.columns :
    if "weighted" in col : table.rename(columns = {col : 'Loan Rates' }, inplace=True)
    elif "period" in col : table.rename(columns = {col : 'date' }, inplace=True)

table['year'] = table['date'].str.extract(r'(\d{4})').ffill()

table['date'] = table['date'].str.replace(".","")
table['month'] = pd.to_datetime(table['date'], format="%b", errors='coerce')
table = table.dropna( subset='month')

table['month'] = table['month'].dt.strftime('%m')
table['date'] = pd.to_datetime(table['year'].astype(str) + '-' + table['month'], format='%Y-%m')
table.set_index("date", inplace=True)

table = table[['Loan Rates']]

df = pd.concat([ df, table ], axis=1)
print(f"Updated Loan Rates: {table.index.max()}")

# Table 8.6
table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "Table 8.6 " in text]
table_2_pages = table_2_pages[0]
page_tables = camelot.read_pdf(file_name, pages=str(table_2_pages), flavor='stream', edge_tol=500, split_text=False, flag_size=True)
table = page_tables[0].df

table.columns = table.iloc[0:3].apply(lambda col: ' '.join(col.astype(str)), axis=0)
table.columns = table.columns.astype(str).str.lower().str.strip()
table.set_index('period', inplace=True)
regions = ['New Providence', 'Grand Bahama', 'Other Family Islands', 'The Bahamas']
categories = ['residential', 'commercial and industrial', 'public', 'total']
new_columns = []
for region in regions:
    for category in categories:
        new_columns.append(f"Construction Permits {region} {category}")

table.columns = new_columns
table = table.reset_index()

table['year'] = table['period'].str.extract(r'(\d{4})').ffill()
table['quarter'] = table['period'].str.lower().str.replace(".","").str.extract(r'(qtr\s?[i|ii|iii|iv]+)')

quarter_map = {'qtr i': 'Q1', 'qtr ii': 'Q2', 'qtr iii': 'Q3', 'qtr iv': 'Q4'}
table['quarter'] = table['quarter'].replace(quarter_map)
table = table.dropna( subset='quarter')
quarter_to_month = {'Q1': 3, 'Q2': 6, 'Q3': 9, 'Q4': 12}
table['month'] = table['quarter'].map(quarter_to_month)

table['date'] = pd.to_datetime(table.assign(day=1)[['year', 'month', 'day']], errors='coerce')
table = table.set_index('date')
table = table[ [ col for col in table.columns if "Construction" in col ] ]

df = pd.concat([ df, table ], axis=1)
print(f"Updated Construction: {table.index.max()}")

# Table 8.8
table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "Table 8.8 " in text]
table_2_pages = table_2_pages[0]

page_tables = camelot.read_pdf(file_name, pages=str(table_2_pages), flavor='stream', edge_tol=500, split_text=False, flag_size=True)
table = page_tables[0].df

table.columns = table.iloc[0:4].apply(lambda col: ' '.join(col.astype(str)), axis=0)
table.columns = table.columns.astype(str).str.lower().str.strip()
table.set_index('period', inplace=True)
regions = ['New Providence', 'Grand Bahama', 'The Bahamas']
categories = ['residential', 'commercial and industrial', 'public', 'total']
new_columns = []
for region in regions:
    for category in categories:
        new_columns.append(f"Construction Starts {region} {category}")

table.columns = new_columns
table = table.reset_index()

table['year'] = table['period'].str.extract(r'(\d{4})').ffill()
table['quarter'] = table['period'].str.lower().str.replace(".","").str.extract(r'(qtr\s?[i|ii|iii|iv]+)')

quarter_map = {'qtr i': 'Q1', 'qtr ii': 'Q2', 'qtr iii': 'Q3', 'qtr iv': 'Q4'}
table['quarter'] = table['quarter'].replace(quarter_map)
table = table.dropna( subset='quarter')
quarter_to_month = {'Q1': 3, 'Q2': 6, 'Q3': 9, 'Q4': 12}
table['month'] = table['quarter'].map(quarter_to_month)

table['date'] = pd.to_datetime(table.assign(day=1)[['year', 'month', 'day']], errors='coerce')
table = table.set_index('date')
table = table[ [ col for col in table.columns if "Construction" in col ] ]

df = pd.concat([ df, table ], axis=1)
print(f"Updated Construction Starts: {table.index.max()}")

# Table 8.10
table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "Table 8.10 " in text]
table_2_pages = table_2_pages[0]

page_tables = camelot.read_pdf(file_name, pages=str(table_2_pages), flavor='stream', edge_tol=500, split_text=False, flag_size=True)
table = page_tables[0].df

table.columns = table.iloc[0:5].apply(lambda col: ' '.join(col.astype(str)), axis=0)
table.columns = table.columns.astype(str).str.lower().str.strip()
table.set_index('period', inplace=True)
regions = ['New Providence', 'Grand Bahama', 'The Bahamas']
categories = ['residential', 'commercial and industrial', 'public', 'total']
new_columns = []
for region in regions:
    for category in categories:
        new_columns.append(f"Construction Completions {region} {category}")

table.columns = new_columns
table = table.reset_index()

table['year'] = table['period'].str.extract(r'(\d{4})').ffill()
table['quarter'] = table['period'].str.lower().str.replace(".","").str.extract(r'(qtr\s?[i|ii|iii|iv]+)')

quarter_map = {'qtr i': 'Q1', 'qtr ii': 'Q2', 'qtr iii': 'Q3', 'qtr iv': 'Q4'}
table['quarter'] = table['quarter'].replace(quarter_map)
table = table.dropna( subset='quarter')
quarter_to_month = {'Q1': 3, 'Q2': 6, 'Q3': 9, 'Q4': 12}
table['month'] = table['quarter'].map(quarter_to_month)

table['date'] = pd.to_datetime(table.assign(day=1)[['year', 'month', 'day']], errors='coerce')
table = table.set_index('date')
table = table[ [ col for col in table.columns if "Construction" in col ] ]

df = pd.concat([ df, table ], axis=1)
print(f"Updated Construction Completions: {table.index.max()}")

# Table 8.17
table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "Table 8.17 " in text]
table_2_pages = table_2_pages[0]

page_tables = camelot.read_pdf(file_name, pages=str(table_2_pages), flavor='stream', edge_tol=500, split_text=False, flag_size=True)
table = page_tables[0].df

table.columns = table.iloc[1:3].apply(lambda col: ' '.join(col.astype(str)), axis=0)
table.columns = table.columns.astype(str).str.lower().str.strip()

table['year'] = table['period'].str.extract(r'(\d{4})').ffill()
table['quarter'] = table['period'].str.lower().str.replace(".","").str.extract(r'(qtr\s?[i|ii|iii|iv]+)')

quarter_map = {'qtr i': 'Q1', 'qtr ii': 'Q2', 'qtr iii': 'Q3', 'qtr iv': 'Q4'}
table['quarter'] = table['quarter'].replace(quarter_map)
table = table.dropna( subset='quarter')
quarter_to_month = {'Q1': 3, 'Q2': 6, 'Q3': 9, 'Q4': 12}
table['month'] = table['quarter'].map(quarter_to_month)

table['date'] = pd.to_datetime(table.assign(day=1)[['year', 'month', 'day']], errors='coerce')
table = table.set_index('date')

for col in table.columns :
    if "generated" in col : table.rename(columns = { col : 'electricity generated' } , inplace=True)
    elif "domestic" in col : table.rename(columns = { col : 'electricity units sold domestic' } , inplace=True)
    elif "commercial" in col : table.rename(columns = { col : 'electricity units sold commercial and industrial' } , inplace=True)
    elif "street" in col : table.rename(columns = { col : 'electricity units sold street lighting' } , inplace=True)
    elif "sales" in col : table.rename(columns = { col : 'electricity units total sales' } , inplace=True)

table = table[[ col for col in table.columns if "electricity" in col ]]
df = pd.concat([ df, table ], axis=1)
print(f"Electricity: {table.index.max()}")

# Table 7.9
table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "Table 7.9 " in text]
table_2_pages = table_2_pages[0]

page_tables = camelot.read_pdf(file_name, pages=str(table_2_pages), flavor='stream', edge_tol=500, split_text=False, flag_size=True)
table = page_tables[0].df

table.columns = table.iloc[0:4].apply(lambda col: ' '.join(col.astype(str)), axis=0)
table.columns = table.columns.astype(str).str.lower().str.strip()

table['year'] = table['period'].str.extract(r'(\d{4})').ffill()
table['quarter'] = table['period'].str.lower().str.replace(".","").str.extract(r'(qtr\s?[i|ii|iii|iv]+)')

quarter_map = {'qtr i': 'Q1', 'qtr ii': 'Q2', 'qtr iii': 'Q3', 'qtr iv': 'Q4'}
table['quarter'] = table['quarter'].replace(quarter_map)
table = table.dropna( subset='quarter')
quarter_to_month = {'Q1': 3, 'Q2': 6, 'Q3': 9, 'Q4': 12}
table['month'] = table['quarter'].map(quarter_to_month)

table['date'] = pd.to_datetime(table.assign(day=1)[['year', 'month', 'day']], errors='coerce')
table = table.set_index('date')

for col in table.columns :
    if "bunkers" in col : table.rename(columns = { col : 'Oil Imports Foreign Bunkers' }, inplace=True)
    elif "total local" in col : table.rename(columns = { col : 'Oil Imports Total Domestic' }, inplace=True)
table = table[ [col for col in table.columns if "Oil Imports" in col ] ]


df = pd.concat([ df, table ], axis=1)
print(f"Oil Imports: {table.index.max()}")

# Table 7.1
table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "Table 7.1 " in text]
table_2_pages = table_2_pages[1]

page_tables = camelot.read_pdf(file_name, pages=str(table_2_pages), flavor='stream', edge_tol=500, split_text=False, flag_size=True)
table = page_tables[0].df

last_head = 5
table.columns = table.iloc[0:last_head].apply(lambda col: ' '.join(col.astype(str)), axis=0)
table.columns = table.columns.astype(str).str.lower().str.strip().str.replace(".","")
table['account'] = table['current account payments'].apply(lambda x: x if 'net acquisition' in x.lower() or x.isupper() else None).ffill()

quarter_map = {'qtr i': 'Q1', 'qtr ii': 'Q2', 'qtr iii': 'Q3', 'qtr iv': 'Q4'}
table = table.iloc[last_head:]
table = table.melt(id_vars=['current account payments', 'account'], var_name='date', value_name='value')
table['current account payments'] = table['current account payments'].astype(str).str.strip().str.lower()
table['date'] = table['date'].astype(str).str.replace("p","")

table['year'] = table['date'].str.extract(r'(\d{4})')
table['quarter'] = table['date'].str.lower().str.replace(".","").str.extract(r'(qtr\s?[i|ii|iii|iv]+)')

quarter_map = {'qtr i': 'Q1', 'qtr ii': 'Q2', 'qtr iii': 'Q3', 'qtr iv': 'Q4'}
table['quarter'] = table['quarter'].replace(quarter_map)

quarter_to_month = {'Q1': 3, 'Q2': 6, 'Q3': 9, 'Q4': 12}
table['month'] = table['quarter'].map(quarter_to_month)

table['date'] = pd.to_datetime(table.assign(day=1)[['year', 'month', 'day']], errors='coerce')
table = table.set_index('date')
table = table[~table.index.isna()]
table = table[['current account payments', 'account', 'value']]

table['account'] = table['account'].astype(str).str.strip().str.lower()
table['current account payments'] = table['current account payments'].str.replace(r'[^a-zA-Z0-9 ]', '', regex=True)
table['account'] = table['account'].str.replace(r'[^a-zA-Z0-9 ]', '', regex=True)
table['variable'] = table['account'] + "_" + table['current account payments']
table['variable'] = table['variable'].str.replace("  ", " ").str.replace(" ", "_")
table['variable'] = table['variable'].str.replace(r'(_omissions).*', r'\1', regex=True)

table = table[['variable', 'value']]

table = table.pivot(columns='variable', values='value')
table = table.apply(pd.to_numeric)

table['Current Acc Balance'] = table[[ col for col in table.columns if "total_receipts" in col ][0]] - table[[ col for col in table.columns if "total_payments" in col ][0]]

df = pd.concat([ df, table ], axis=1)
print(f"Balance of Payments: {table.index.max()}")

# Transform all variables in numeric and save.
for col in df.columns : df[col] = pd.to_numeric(df[col].astype(str).str.strip().str.replace(",",""), errors='coerce')

print(f"Updated: {df.index.max()}")
df.to_csv(f'{PATH_RAW}/bahamas qsd.csv')

try :
    os.remove(file_name)
except :
    True

## CBOB/MEFD

In [ ]:
# Open website
url = "https://www.centralbankbahamas.com/publications/monthly-economic-and-financial-development-report"
r = requests.get(url=url, verify=None).content
soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links = [link for link in links if "pdf" in link]
#links = [link for link in links if "quarterly-statistical" in link]
links = list(set(links))
links = [ link for link in links if "MEFD" in link ]

# Extract the date from the file name
from datetime import datetime
def extract_date(url):
    # Extract the date from the URL string (the part before "-MEFD")
    date_str = url.split('/')[-1].split('/')[-1][:10]
    return datetime.strptime(date_str, '%Y-%m-%d')
latest_url = max(links, key=extract_date)
links = [ latest_url ]

df = pd.DataFrame()

# Iterate across all links.
for link in links :
    #link = links[0]
    # Each link is a pdf.
    # Download the document.
    print(f"Latest: {link}")
    response = requests.get( link, verify=False, timeout=300, headers=hdr)
    file_name = link.split("/")[-1]
    file_name = f"{folder_spec}/{file_name}"
    with open( file_name , 'wb') as file: file.write(response.content)
    print(f"Downloaded")

    # Extract the pdf text
    page_texts = extract_page_text(file_name)
    
    # Get the page
    table_2_pages = [i + 1 for i, text in enumerate(page_texts) if "Summary Accounts of the Central Bank" in text]
    table_2_pages = table_2_pages[0]

    # Read the pdf as a table
    page_tables = camelot.read_pdf(file_name, pages=str(table_2_pages), flavor='stream', edge_tol=500, split_text=True, flag_size=True)
    table = page_tables[0].df

    # Find the header names using the word "VALUE"
    if 'VALUE' in ''.join(list(table.iloc[0].values)): table.columns = table.iloc[1]
    else: table.columns = table.iloc[0]
    table.rename( columns = { '' : 'variable' }, inplace=True)
    table = table[table['variable']!=""]

    # Reshape
    table = pd.melt(table, id_vars="variable", var_name="date", value_name='value')

    # Extract the date of the file name
    match = re.search(r'(\d{4})-\d{2}-\d{2}-\d{2}-\d{2}-\d{2}-MEFD-[A-Za-z]+-(\d{4})\.pdf', file_name)
    if match is None : match = re.search(r'(\d{4})-\d{2}-\d{2}-\d{2}-\d{2}-\d{2}.*MEFD.*(\d{4})\.pdf', file_name)
    year = match.group(2)

    # Working with dates: Each column has the month (year is in the pdf name).
    # If the column names say "January" then December dates are pre-January.
    if table['date'].str.contains('Jan').any() :
        table['date'] = table['date'].apply( lambda x: f"{year} {str(x)}" if 'Jan' in str(x) else f"{int(year) - 1} {str(x)}" if 'Dec' in str(x) else x)
    else :
        table['date'] = table['date'].apply( lambda x: f"{year} {str(x)}" )
    table['date'] = pd.to_datetime( table['date'].str.replace(".","").str.lower() , format = "%Y %b %d", errors = 'coerce')
    table.set_index('date' , inplace=True)

    # Clean numbers
    table['value'] = table['value'].str.replace(",", "", regex=False)
    table['value'] = table['value'].str.replace(r"\((.*?)\)", r"-\1", regex=True)
    table['value'] = pd.to_numeric(table['value'], errors='coerce')
    
    table['variable'] = table['variable'].str.replace(r'\(.*?\)', '', regex=True)
    table['variable'] = table['variable'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)
    
    table = table.groupby(['date', 'variable']).first().reset_index().set_index('date')
    table = table.pivot(columns='variable', values='value')
    table = table.sort_index()
    
    table.columns = table.columns.str.lower().str.strip()
    
    df = pd.concat( [ df, table] , axis=1)
    df.columns = df.columns.str.replace("  ", " ")
    df = df.groupby(axis=1, level=0).first()
    try :
        os.remove(file_name)
    except :
        True

for col in df.columns : df[col] = pd.to_numeric(df[col].astype(str).str.strip().str.replace(",",""), errors='coerce')

print(f"Updated: {df.index.max()}")
df.to_csv(f'{PATH_RAW}/bahamas mefd.csv')

# INTERNATIONAL


## USA

In [ ]:
'''
This code extracts data from the Federal Reserve
an API Key is required
Expand the data by adding  any desired series to the series list
'''
series = ["EXCAUS", "DEXUSEU", "DEXUSUK", "SP500", "DJIA", "NASDAQCOM"]

df = pd.DataFrame()
for serie in series :
    #&realtime_end=9999-12-31
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={serie}&api_key={api_key}&file_type=json"
    response = requests.get(url)
    response = response.json()
    dfi = pd.DataFrame(response['observations'])
    dfi = dfi[['value', 'date']]
    dfi.set_index('date', inplace=True)
    dfi.index = pd.to_datetime(dfi.index)
    dfi.rename(columns = { "value" : f"fred_{serie}" }, inplace=True)
    dfi[f"fred_{serie}"] = dfi[f"fred_{serie}"].apply(pd.to_numeric, errors='coerce')
    df = df.merge(dfi, left_index=True, right_index=True, how='outer')
    df = pd.DataFrame(df.resample("D").mean())
    #df = pd.concat([df, dfi], axis=1)
    df.sort_index(inplace=True)
    
df.to_csv(f"{PATH_RAW}/fred_data.csv")
print(f"FRED Updated: {df.index.max()}")


'''
Access the EIA and download the xls file that has all data.
'''
url = "https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?n=PET&s=RBRTE&f=D"

r = requests.get(url=url, verify=None).content
soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
link = [ link for link in links if "xls" in link ][0].replace("../","")
get = f"https://www.eia.gov/dnav/pet/{link}"
df = pd.ExcelFile(get)

df = pd.read_excel( df , sheet_name=[ sheet for sheet in df.sheet_names if "data" in sheet.lower() ][0], skiprows=2)
df.columns = df.columns.str.lower()
df.set_index('date', inplace=True)
df = df.rename(columns=lambda x: "brent price" if "brent" in x else x)
df.index = pd.to_datetime(df.index)

df.to_csv(f"{PATH_RAW}/eia_brent.csv")
print(f"EIA  Updated: {df.index.max()}")

In [ ]:
from io import BytesIO
'''
Access the US Census and dowload the xls file with all the data
'''

url = "https://www.census.gov/foreign-trade/balance/c2360.html"

r = requests.get(url=url, verify=None).content
soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
get = [ link for link in links if "xls" in link ]
#get = f"https://www.census.gov/{get[0]}"

get = "https://www.census.gov/foreign-trade/balance/country.xlsx"
r = requests.get(get, headers=hdr)
df = pd.read_excel(BytesIO(r.content), engine="openpyxl")


In [39]:
df = df[df['CTYNAME'].str.contains("Bahamas")]

df = df.drop(['CTY_CODE', 'CTYNAME', 'IYR', 'EYR'], axis=1)
df.columns = df.columns.str.lower()
df = pd.melt(df, id_vars="year", var_name="month", value_name='trade')

df['variable'] = df['month'].str[0]
df['month'] = df['month'].str[1:]
df['variable'] = df['variable'].replace({'i': 'exports USA', 'e': 'imports USA'})

df['date'] = pd.to_datetime(df['year'].astype(str)+df['month'], format="%Y%b") + pd.offsets.MonthEnd(0)
df = df.set_index('date')[['trade', 'variable']]

df = df.pivot(columns='variable', values='trade')
df = df[df.index <= df[df["exports USA"] != 0].index[-1]]

df.to_csv(f"{PATH_RAW}/uscensus_trade.csv")
print(f"US Census Updated: {df.index.max()}")

US Census Updated: 2025-10-31 00:00:00


## Canada

In [ ]:
#conda install conda-forge::zipfile2
'''
This part of the script accessess the canadian open data site.
It downloads the respective zip file and processes data.
'''

import zipfile
import io
import shutil
def extract_year(url):
    match = re.search(r'(20\d{2})', url)
    if match:
        return int(match.group(1))
    return None

df = pd.read_csv(f"{PATH_RAW}/canada_imports.csv", index_col='date', parse_dates=True)
#df = pd.DataFrame()
url = "https://open.canada.ca/data/en/dataset/2909a648-5753-4924-878a-b069392d9cde"

r = requests.get(url=url, verify=None).content
soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links = [ link for link in links if "Imp" in link ]
links = [ link for link in links if extract_year(link.split("/")[-1]) >= df.index.max().year ]

for link in links : 
    # Download the Zip file
    response = requests.get(link)
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        z.extractall(f"{PATH_RAW}")  # Extract to a folder
        print("ZIP extracted successfully!")

    # Extract file 022 (imports by product)
    file_name = [ x for x in z.namelist() if "022" in x ][0]
    file_name = f"{PATH_RAW}/{file_name}"
    # Open the file
    dfi = pd.read_csv(file_name)

    # Clean variable names
    for col in dfi.columns :
        if "YearMonth" in col :
            dfi = dfi.rename(columns = {col : 'date'} )
        elif "Country" in col :
            dfi = dfi.rename(columns = {col : 'partner'} )
        elif "Value" in col :
            dfi = dfi.rename(columns = {col : 'value'} )
        elif "Quantity" in col :
            dfi = dfi.rename(columns = {col : 'quantity'} )
        elif "Unit" in col :
            dfi = dfi.rename(columns = {col : 'unit'} )

    # Find Bahamas
    dfi = dfi[dfi['partner']=='BS']
    # Clean date
    dfi['date'] = pd.to_datetime(pd.to_datetime(dfi['date'] , format="%Y%m").dt.strftime('%Y-%m-%d'))
    dfi = dfi.set_index('date')

    # Sum all monthly values
    dfi = dfi.resample('MS')[['value']].sum()
    dfi.rename(columns = {"value" : 'imports_can'} , inplace=True)
    df = df[~df.index.isin(dfi.index.unique())]
    df = pd.concat([df, dfi], axis=0)

    try :
        folder_path = os.path.dirname(os.path.abspath(file_name))
        if os.path.exists(folder_path): shutil.rmtree(folder_path)
    except :
        continue

print(f"Updated: {df.index.max()}")
df.to_csv(f'{PATH_RAW}/canada_imports.csv')

# EXPORTS
df = pd.read_csv(f"{PATH_RAW}/canada_exports.csv", index_col='date', parse_dates=True)

links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links = [ link for link in links if "tot_exp" in link.lower() ]
links = [ link for link in links if extract_year(link.split("/")[-1]) >= df.index.max().year ]

for link in links : 
    response = requests.get(link)
    
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        z.extractall(f"{PATH_RAW}")  # Extract to a folder
        print("ZIP extracted successfully!")
        
    file_name = [ x for x in z.namelist() if "021" in x ][0]
    file_name = f"{PATH_RAW}/{file_name}"
    dfi = pd.read_csv(file_name)
    for col in dfi.columns :
        if "YearMonth" in col :
            dfi = dfi.rename(columns = {col : 'date'} )
        elif "Country" in col :
            dfi = dfi.rename(columns = {col : 'partner'} )
        elif "Value" in col :
            dfi = dfi.rename(columns = {col : 'value'} )
        elif "Quantity" in col :
            dfi = dfi.rename(columns = {col : 'quantity'} )
        elif "Unit" in col :
            dfi = dfi.rename(columns = {col : 'unit'} )
    dfi = dfi[dfi['partner']=='BS']
    dfi['date'] = pd.to_datetime(pd.to_datetime(dfi['date'] , format="%Y%m").dt.strftime('%Y-%m-%d'))
    dfi = dfi.set_index('date')
    dfi = dfi.resample('MS')[['value']].sum()
       
    dfi.rename(columns = {"value" : 'exports_can'} , inplace=True)
    df = df[~df.index.isin(dfi.index.unique())]
    df = pd.concat([df, dfi], axis=0)

    try :
        folder_path = os.path.dirname(os.path.abspath(file_name))
        if os.path.exists(folder_path): shutil.rmtree(folder_path)
    except :
        continue

print(f"Updated: {df.index.max()}")
df.to_csv(f'{PATH_RAW}/canada_exports.csv')

## EU

In [7]:
'''
Download data from eurostat.
eurostat package is required.
'''
# conda install conda-forge::eurostat
import eurostat
dataset = 'ds-045409'
params = {
    'format': 'TSV',        # Use uncompressed TSV format for simplicity
    'geo': 'EU',            # Specify the region (EU)
    'partner': 'BS',        # Specify the partner (Bahamas)
    'time': '2021',         # You can specify a smaller time range (e.g., 2021)
    'trade': '1',           # You can filter for imports (1) or exports (2)
    'product': '0',         # You can specify product codes if needed
    'compressed': 'false',  # Disable compression
}
df = eurostat.get_data_df(dataset,
                       filter_pars = {
                           'reporter' : 'EU' ,
                           'partner' : 'BS' ,
                           'indicators' : 'VALUE_IN_EUROS',
                           'freq' : 'M' ,
                           'product' : 'TOTAL'
                       })
df.drop(['freq', 'reporter', 'partner', 'product'], axis=1, inplace=True)

df = pd.melt(df,
        id_vars=['flow', 'indicators\\TIME_PERIOD'],
        var_name='date', value_name='value')

df['date'] = pd.to_datetime(pd.to_datetime(df['date'] , format="%Y-%m").dt.strftime('%Y-%m-%d'))
df['flow'] = df['flow'].map({"1": 'imp_eu', "2": 'exp_eu'})
df.set_index('date', inplace=True)
df = df.pivot_table(index=df.index, columns='flow', values='value')

print(f"Updated: {df.index.max()}")
df.to_csv(f'{PATH_RAW}/eu_trade.csv')

Updated: 2025-11-01 00:00:00


## UK

In [8]:
# Prepare all functions to read data.

import pprint as pp
import logging

logger = logging.getLogger(__name__)

ROOT_URL = "https://api.beta.ons.gov.uk/v1/"


def get_list_of_datasets():
    """Get list of all datasets available from API.
    Currently (August 2021), there are 41 available datasets.

    Returns
    -------
    list of dicts
        Metadata objects for each available dataset.
    """
    num_to_get = 100
    datasets = []

    offset = 0
    while len(datasets) < num_to_get:
        r = requests.get(ROOT_URL + "datasets", params={"offset": offset})
        results = r.json()
        [logger.info(item.get("title")) for item in results.get("items")]
        datasets.extend(results.get("items"))
        num_retrieved = results.get("count")
        offset += num_retrieved
        if num_retrieved == 0:
            break
    logger.info(f"\nFound {len(datasets)} datasets")
    return datasets


def get_dataset_by_name(datasets, target_name):
    """Get a dataset by matching on its title (or part of the title).
    Returns the first dataset object whose name contains the given target_name string.

    Parameters
    ----------
    datasets : List
        List of dataset objects
    target_name : str
        name (or partial name) of target dataset

    Returns
    -------
    dict
        Dataset object (or None, if no match is found)
    """
    for ds in datasets:
        if target_name.lower() in ds.get("title").lower():
            logger.info(f"Found dataset '{ds.get('title')}'")
            return ds
    logger.info(f"No dataset found containing '{target_name}'")
    return None


def get_edition(dataset, prefered_edition="time-series"):
    """Get one edition of a dataset. If no preferred edition is
    specified, return the most recent one.

    Parameters
    ----------
    dataset : dict
        dataset metadata
    prefered_edition : str, optional
        name of edition, by default "time-series"

    Returns
    -------
    str
        URL of edition
    """
    editions_url = dataset.get("links").get("editions").get("href")
    r = requests.get(editions_url)
    results = r.json()
    for row in results.get("items"):
        if row.get("edition") == prefered_edition:
            edition = row.get("links").get("latest_version").get("href")
            return edition

    # Default to latest version, if requested version is not found.
    latest_version = dataset.get("links").get("latest_version").get("href")
    return latest_version


def get_dimensions(edition_url):
    """Builds dictionary of all valid options for all dimensions of a given dataset,
    with descriptions for each option.
    Individual obvserviations can later be obtained by choosing from these options.
    Ranges can later be obtained by replacing one dimesion with the wildcard '*'.

    Parameters
    ----------
    dataset : dict
        single dataset

    Returns
    -------
    dict of dicts
        map of {dimensions:{acceptable_values:description}}
    """
    valid_dimensions = {}
    r = requests.get(edition_url + "/dimensions")
    results = r.json()
    for dimension in results.get("items"):
        logger.info(f'{dimension.get("name")}: \t{dimension.get("label")}')
        dim_id = dimension.get("links").get("options").get("id")
        options_url = f"{edition_url}/dimensions/{dim_id}/options"

        sr = requests.get(options_url, params={"limit": 50})
        sresults = sr.json()
        # TODO! Could add in paging here, as there *could* be multiple pages of valid options.
        logger.info(f"\tHas {sresults.get('count')} options")
        # valid_options = [item.get("option") for item in sresults.get("items")]
        option_descriptions = {
            item.get("option"): item.get("label") for item in sresults.get("items")
        }
        logger.info(f'{dimension.get("name")}: {option_descriptions}')
        valid_dimensions[dimension.get("name")] = option_descriptions

    return valid_dimensions


def choose_dimensions(valid_dims, overrides={}):
    """For each dimension, choose a single valid option (except for 'time', where
    we use the wildcard '*' to get the whole time-series.)
    If not specified, choose the first valid option for each dimension.

    Parameters
    ----------
    valid_dims : dict
        map of lists of valid dimension values
    overrides : dict, optional
        selected dimensions

    Returns
    -------
    dict
        final choice of dimensions
    """
    # By default, choose first valid item for all dimensions; then override where needed:
    chosen_dimensions = {k: next(iter(v.keys())) for k, v in valid_dims.items()}
    # get whole time-series, not just a single point in time:
    chosen_dimensions["time"] = "*"
    chosen_dimensions.update(overrides)
    return chosen_dimensions


def get_observations(edition_url, dimensions):
    """[summary]

    Parameters
    ----------
    edition_url : str
        URL of this edition of the data
    dimensions : dict
        dimensions specifying slice of data required

    Returns
    -------
    pd.Dataframe
        Summary of data, with columns "id" (time) and "observation" (value)
    """
    r = requests.get(edition_url + "/observations", params=dimensions)
    results = r.json()
    summary = []
    for observation in results.get("observations"):
        id = observation.get("dimensions").get("Time").get("id")
        summary.append({"id": id, "observation": observation.get("observation")})
    df = pd.DataFrame(summary)
    return df


def get_timeseries(dataset_name, dimension_values):
    """Get a specified dataset time-series, with a given set of dimensions.
    NB: dataframe is not sorted.

    Parameters
    ----------
    dataset_name : str
        Descriptive name of data set.
    dimension_values : dict
        set of valid dimensions for this dataset.
        If set to "None", then return set of valid dimesions.

    Returns
    -------
    Either:
        dict
            Set of valid dimensions, if None specified in function call
    or:
        dataframe
            containing time series
        dict
            dataset metadata
        str
            url of this edition of the data
    """
    dss = get_list_of_datasets()
    ds = get_dataset_by_name(dss, dataset_name)
    edition_url = get_edition(ds)
    valid_dims = get_dimensions(edition_url)
    logger.info(valid_dims)
    if dimension_values is None:
        return valid_dims
    chosen_dimensions = choose_dimensions(valid_dims, dimension_values)
    df = get_observations(edition_url, chosen_dimensions)
    logger.info(df.shape)
    return df, ds, edition_url


def demo():
    print("=" * 70)
    print("List of available datasets:")
    dss = get_list_of_datasets()
    [print(item.get("title")) for item in dss]

    print("=" * 70)
    dataset_name = "UK Labour Market"
    print(f"Valid options for dimensions for the {dataset_name}, with descriptions")
    # Get the set of valid dimensions for the Labour Market set.
    # E.g. list of valid age groups, economic activity categories etc.

    dimensions = get_timeseries(dataset_name, None)
    pp.pprint(dimensions)
    print("\n")

    # We've now selected specific dimensions for our request:
    labour_market_dimensions = {
        "economicactivity": "in-employment",
        "agegroups": "16+",
        "seasonaladjustment": "seasonal-adjustment",
        "sex": "all-adults",
        "unitofmeasure": "rates",
    }
    print(f"Chosen dimensions for the {dataset_name}")
    pp.pprint(labour_market_dimensions, indent=4)

    df_labour = get_timeseries(dataset_name, labour_market_dimensions)[0]
    df_labour["year"] = (
        df_labour["id"].str[-4:].astype(int)
    )  # extract year as last 4 digits of row id
    df_labour = df_labour.sort_values("year")
    print("")
    print(df_labour)
    print("\n")

    # Repeat the process for a second time series, GDP, with specific dimensions:
    print("=" * 70)
    gdp_dataset_name = "annual GDP"
    gdp_dimensions = {
        "geography": "UK0",
        "unofficialstandardindustrialclassification": "A--T",
    }
    print(f"Chosen dimensions for the {dataset_name}")
    df_gdp = get_timeseries(gdp_dataset_name, gdp_dimensions)[0]
    df_gdp = df_gdp.sort_values("id")
    pp.pprint(gdp_dimensions, indent=4)
    print("")
    print(gdp_dataset_name)
    print(df_gdp)

    print("=" * 70)
    print("End of demo!")

In [9]:
#dss = get_list_of_datasets()
dataset_name = "Trade in goods: country by commodity"
#dss = [ item for item in dss if item.get("title")==dataset_name ]
#dimensions = get_timeseries(dataset_name, None)
#dimensions

df = pd.DataFrame()

for var in ['exp_uk' , 'imp_uk'] :
    if var == "exp_uk" : t = 'EX'
    elif var == "imp_uk" : t = 'IM'
    params = {
        "countriesandterritories": "BS",
        "standardindustrialtradeclassification": "T",
        #"standardindustrialtradeclassification": "0",
        'geography': 'K02000001' ,
        'direction': t ,
        }
    dfi = get_timeseries(dataset_name, params)
    dfi = dfi[0]
    dfi['date'] = pd.to_datetime(dfi['id'], format="%b-%y")
    dfi = dfi.set_index('date')
    dfi.rename(columns = { 'observation' : var} , inplace=True)
    dfi = dfi[[var]]
    dfi[var] = pd.to_numeric(dfi[var])
    
    df = pd.concat( [df, dfi], axis=1)

print(f"Updated: {df.index.max()}")
df.to_csv(f'{PATH_RAW}/uk_trade.csv')

Updated: 2025-11-01 00:00:00
